# Forward Pass from Scratch

Companion notebook for Section 4 of the MLP lecture notes.

Objectives:

- reproduce the 2-2-1 forward-pass example from the notes;
- distinguish pre-activations `z` from activations `a`;
- implement the computation scalar-by-scalar and with matrices;
- extend the same calculation to a mini-batch;
- connect the notebook with the HTML forward-pass simulator.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (7, 4)
plt.rcParams['axes.grid'] = True
np.set_printoptions(precision=6, suppress=True)


## 1. Parameters from the lecture notes

The network has two input features, one hidden layer with two sigmoid neurons, and one sigmoid output neuron.


In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

x = np.array([0.05, 0.10])
y_true = 0.01

W1 = np.array([
    [0.15, 0.20],
    [0.25, 0.30],
])
b1 = np.array([0.35, 0.35])

W2 = np.array([[0.40, 0.45]])
b2 = np.array([0.60])

print('x:', x)
print('W1 shape:', W1.shape)
print('b1 shape:', b1.shape)
print('W2 shape:', W2.shape)
print('b2 shape:', b2.shape)


## 2. Scalar computation

First compute each neuron explicitly. This matches the arithmetic shown in the notes.


In [ ]:
z1_1 = 0.15 * x[0] + 0.20 * x[1] + 0.35
z1_2 = 0.25 * x[0] + 0.30 * x[1] + 0.35

a1_1 = sigmoid(z1_1)
a1_2 = sigmoid(z1_2)

z2_1 = 0.40 * a1_1 + 0.45 * a1_2 + 0.60
y_hat = sigmoid(z2_1)

print('z1_1:', z1_1)
print('z1_2:', z1_2)
print('a1_1:', a1_1)
print('a1_2:', a1_2)
print('z2_1:', z2_1)
print('y_hat:', y_hat)


## 3. Matrix computation

The same forward pass is compactly written as:

`z1 = W1 @ x + b1`

`a1 = sigmoid(z1)`

`z2 = W2 @ a1 + b2`

`a2 = sigmoid(z2)`


In [ ]:
z1 = W1 @ x + b1
a1 = sigmoid(z1)
z2 = W2 @ a1 + b2
a2 = sigmoid(z2)

forward_table = pd.DataFrame({
    'quantity': ['z1[0]', 'z1[1]', 'a1[0]', 'a1[1]', 'z2[0]', 'a2[0]'],
    'value': [z1[0], z1[1], a1[0], a1[1], z2[0], a2[0]],
})
forward_table


In [ ]:
print('Scalar and matrix outputs match?', np.allclose(y_hat, a2[0]))
print('Prediction rounded to 4 decimals:', round(float(a2[0]), 4))


## 4. Loss for this prediction

The notes use mean squared error with a factor of `1/2`: `0.5 * (y_hat - y)^2`.


In [ ]:
loss = 0.5 * (a2[0] - y_true) ** 2
print('y_true:', y_true)
print('y_hat:', a2[0])
print('MSE-style loss:', loss)


## 5. Store intermediate values for backpropagation

The backward pass needs stored values from the forward pass: pre-activations, activations, inputs, prediction, and loss.


In [ ]:
cache = {
    'x': x,
    'z1': z1,
    'a1': a1,
    'z2': z2,
    'a2': a2,
    'y_true': y_true,
    'loss': loss,
}

for key, value in cache.items():
    print(f'{key}: {value}')


## 6. Mini-batch forward pass

For a batch `X` with shape `(batch_size, n_features)`, it is convenient to use scikit-learn-style row-major data:

`Z1 = X @ W1.T + b1`

`A1 = sigmoid(Z1)`

`Z2 = A1 @ W2.T + b2`


In [ ]:
X_batch = np.array([
    [0.05, 0.10],
    [0.00, 1.00],
    [1.00, 0.00],
    [1.00, 1.00],
])

def forward_batch(X):
    Z1 = X @ W1.T + b1
    A1 = sigmoid(Z1)
    Z2 = A1 @ W2.T + b2
    A2 = sigmoid(Z2)
    return Z1, A1, Z2, A2

Z1, A1, Z2, A2 = forward_batch(X_batch)

print('X_batch shape:', X_batch.shape)
print('Z1 shape:', Z1.shape)
print('A1 shape:', A1.shape)
print('Z2 shape:', Z2.shape)
print('A2 shape:', A2.shape)

pd.DataFrame(A2, columns=['prediction'])


## 7. Visualizing the computation graph values

The same quantities appear in the lecture figure: inputs, hidden pre-activations and activations, output pre-activation, prediction, and loss.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.8))
ax.axis('off')

positions = {
    'x1': (0.05, 0.75),
    'x2': (0.05, 0.25),
    'h1': (0.45, 0.75),
    'h2': (0.45, 0.25),
    'out': (0.85, 0.50),
}
labels = {
    'x1': f'x1={x[0]:.2f}',
    'x2': f'x2={x[1]:.2f}',
    'h1': f'z1={z1[0]:.4f}\na1={a1[0]:.4f}',
    'h2': f'z2={z1[1]:.4f}\na2={a1[1]:.4f}',
    'out': f'z={z2[0]:.4f}\ny_hat={a2[0]:.4f}',
}

for name, (px, py) in positions.items():
    circle = plt.Circle((px, py), 0.09, color='white', ec='black', lw=2)
    ax.add_patch(circle)
    ax.text(px, py, labels[name], ha='center', va='center', fontsize=9)

for start in ['x1', 'x2']:
    for end in ['h1', 'h2']:
        ax.annotate('', xy=positions[end], xytext=positions[start], arrowprops=dict(arrowstyle='->', lw=1.2))
for start in ['h1', 'h2']:
    ax.annotate('', xy=positions['out'], xytext=positions[start], arrowprops=dict(arrowstyle='->', lw=1.2))

ax.text(0.5, 0.04, f'loss = 0.5 * (y_hat - y)^2 = {loss:.4f}', ha='center', fontsize=11)
ax.set_title('Forward pass values for the 2-2-1 network')
plt.show()


## 8. Interactive simulator

The repository also contains an HTML simulator for this same concept: `anim/mlp_forward_simulator.html`. Open it in a browser to edit inputs, weights, and biases interactively.


In [ ]:
from pathlib import Path

candidate_paths = [
    Path('anim/mlp_forward_simulator.html'),
    Path('../../anim/mlp_forward_simulator.html'),
]
existing = [path for path in candidate_paths if path.exists()]

print('Repository path: anim/mlp_forward_simulator.html')
print('Found simulator from current working directory?', bool(existing))
if existing:
    print('Resolved path:', existing[0])


## Exercises

1. Change `b2` from 0.60 to 0.00. How much does the prediction change?
2. Replace sigmoid in the hidden layer with ReLU. Does the output increase or decrease?
3. Add a third hidden neuron. What shapes should `W1`, `b1`, and `W2` have?


## Takeaways

- A forward pass alternates linear transformations and activations.
- Pre-activations `z` and activations `a` are different quantities.
- Intermediate values must be stored because backpropagation reuses them.
- Batch computations use the same formulas with matrices.
